# Exploring Training Sizes

The goal of this experiment is to undestrand how the training size affects the performance of the model. We will train the model in the following number of samples by model: (2k, 3k, 5k, 10k, 15k and 20k). We will use the same collections for all models but with random choice.

In [1]:
import polars as pl
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import gc
import torch
import types
import sys
import os
import json
import joblib
import random
from pathlib import Path


from sklearn.linear_model import LogisticRegression
from tqdm import tqdm
from numpy import float64
from sklearn.metrics import roc_auc_score
from utils.metrics.calculate_metric import calculate_agg_metric
from utils.set_random_seed import set_random_seed
from interpret.glassbox import ExplainableBoostingClassifier
from utils.model_loaders import load_logistic_models_for_subfolder



set_random_seed(42)
EXPERIMENTS = ["experiment_1"]

INFO 03-19 16:37:02 [__init__.py:216] Automatically detected platform cuda.


In [21]:
def _get_split_sets(df, subfolder: str, split: str, num_collections: int):
    
    if split == "train":
        TOTAL_NUM_COLLECTIONS = 20000
    else:
        TOTAL_NUM_COLLECTIONS = 200
    
    collections = random.sample(range(TOTAL_NUM_COLLECTIONS), num_collections)
    split_df = df.filter((pl.col("subfolder") == subfolder) & (pl.col("split") == split)).filter(pl.col("collection_idx").is_in(collections))
    X_split = []
    y_split = []

    for exp in EXPERIMENTS:
        exp_df = split_df.filter(pl.col("experiment") == exp)

        X_exp = exp_df.select("input").to_numpy()
        X_exp = np.array([i[0] for i in X_exp])
        y_exp = exp_df.select("evaluation").to_numpy()

        # Data is interleaved: first 500 elements go to position 0 of each array,
        # next 500 elements go to position 1 of each array, etc.
        num_arrays = 500

        X_exp_reshaped = []
        y_exp_reshaped = []
        for i in range(num_arrays):
            # Take every 500th element starting from index i
            X_exp_reshaped.append(X_exp[i::num_arrays])
            y_exp_reshaped.append(y_exp[i::num_arrays])

        X_split.append(X_exp_reshaped)
        y_split.append(y_exp_reshaped)

    return X_split, y_split


def get_train_sets(df, subfolder, num_collections):
    return _get_split_sets(df, subfolder=subfolder, split="train", num_collections=num_collections)


def get_test_sets(df, subfolder):
    return _get_split_sets(df, subfolder=subfolder, split="test", num_collections=200)

In [5]:
df = []

for subfolder in ["voting_alt1", "groundtruth"]:
    for exp in EXPERIMENTS:
        for split in ["train", "test"]:
            try:
                load_df = pl.read_ipc(f"../binary_collections/{subfolder}/{exp}/{split}.feather")
                load_df = load_df.with_columns([
                    pl.lit(subfolder).alias("subfolder"),
                    pl.lit(exp).alias("experiment"),
                    pl.lit(split).alias("split"),
                ])
                df.append(load_df)
            except Exception as e:
                print(f"Could not load {subfolder}/{exp}/{split}: {e}")
df = pl.concat(df)
df.head()

Could not memory_map compressed IPC file, defaulting to normal read. Toggle off 'memory_map' to silence this warning.
Could not memory_map compressed IPC file, defaulting to normal read. Toggle off 'memory_map' to silence this warning.
Could not memory_map compressed IPC file, defaulting to normal read. Toggle off 'memory_map' to silence this warning.
Could not memory_map compressed IPC file, defaulting to normal read. Toggle off 'memory_map' to silence this warning.


collection_idx,test_idx,input,evaluation,subfolder,experiment,split
i64,i64,"array[i64, 100]",i32,str,str,str
0,0,"[0, 1, … 0]",1,"""voting_alt1""","""experiment_1""","""train"""
0,1,"[0, 1, … 0]",1,"""voting_alt1""","""experiment_1""","""train"""
0,2,"[0, 1, … 0]",1,"""voting_alt1""","""experiment_1""","""train"""
0,3,"[0, 1, … 0]",0,"""voting_alt1""","""experiment_1""","""train"""
0,4,"[0, 1, … 0]",1,"""voting_alt1""","""experiment_1""","""train"""


In [ ]:
## Load best param


best_params = {
    "judge": [],
    "groundtruth": []
}

for f in sorted(os.listdir("../best_params")):
    _params = []
    results = json.load(open(f"../best_params/{f}", "r"))["results"]
    for p in results:
        try:
            _params.append(p["best_params"])
        except KeyError:
            _params.append(None)
    
    if "judge" in f:
        best_params["judge"].append(_params)
    else:
        best_params["groundtruth"].append(_params)

In [6]:
judge = df.filter(pl.col("subfolder")=="voting_alt1").filter(pl.col("split")=="train")
judge.group_by("evaluation").agg(pl.count()).sort("evaluation")

/tmp/ipykernel_705426/4011105780.py:2: DeprecationWarning: `pl.count()` is deprecated. Please use `pl.len()` instead.
(Deprecated in version 0.20.5)
  judge.group_by("evaluation").agg(pl.count()).sort("evaluation")


evaluation,count
i32,u32
0,4456964
1,5543036


In [7]:
judge = df.filter(pl.col("subfolder")=="voting_alt1").filter(pl.col("split")=="test")
judge.group_by("evaluation").agg(pl.count()).sort("evaluation")

/tmp/ipykernel_705426/3648853721.py:2: DeprecationWarning: `pl.count()` is deprecated. Please use `pl.len()` instead.
(Deprecated in version 0.20.5)
  judge.group_by("evaluation").agg(pl.count()).sort("evaluation")


evaluation,count
i32,u32
0,64893
1,35107


In [6]:
gt = df.filter(pl.col("subfolder")=="groundtruth").filter(pl.col("split")=="train")
gt.group_by("evaluation").agg(pl.count()).sort("evaluation")

/tmp/ipykernel_711609/3606013892.py:2: DeprecationWarning: `pl.count()` is deprecated. Please use `pl.len()` instead.
(Deprecated in version 0.20.5)
  gt.group_by("evaluation").agg(pl.count()).sort("evaluation")


evaluation,count
i32,u32
0,6045521
1,3954479


In [7]:
gt = df.filter(pl.col("subfolder")=="groundtruth").filter(pl.col("split")=="test")
gt.group_by("evaluation").agg(pl.count()).sort("evaluation")

/tmp/ipykernel_711609/2354382005.py:2: DeprecationWarning: `pl.count()` is deprecated. Please use `pl.len()` instead.
(Deprecated in version 0.20.5)
  gt.group_by("evaluation").agg(pl.count()).sort("evaluation")


evaluation,count
i32,u32
0,60765
1,39235


## 3K Collections Training

In [19]:
X_train_judge_3k, y_train_judge_3k = get_train_sets(df, subfolder="voting_alt1", num_collections=3000)
X_train_groundtruth_3k, y_train_groundtruth_3k = get_train_sets(df, subfolder="groundtruth", num_collections=5000)

judge_models = []
groundtruth_models = []




with tqdm(total=1 * 500, desc="Training models") as pbar:
    for i in range(1):
        _j_arr, _g_arr = [], []
        
        for j in range(500):
            # # Judge model
            # X = X_train_judge_3k[i][j]
            # y = y_train_judge_3k[i][j].ravel()


            # if best_params["judge"][i][j] is None:
            #     _j_arr.append(None)
            # else:
            #     model = LogisticRegression(**best_params["judge"][i][j], max_iter=1000, random_state=42)
            #     model.fit(X, y)
            #     _j_arr.append(model)

            ## Groundtruth model
            X = X_train_groundtruth_3k[i][j]
            y = y_train_groundtruth_3k[i][j].ravel()

            if best_params["groundtruth"][i][j] is None or y.sum() == 0:
                _g_arr.append(None)
            else:
                model = LogisticRegression(**best_params["groundtruth"][i][j], max_iter=100, random_state=42)
                model.fit(X, y)
                _g_arr.append(model)
            
            pbar.update(1)
        
        # judge_models.append(_j_arr)
        groundtruth_models.append(_g_arr)

del X_train_judge_3k, y_train_judge_3k, X_train_groundtruth_3k, y_train_groundtruth_3k
gc.collect()

Training models:   3%|█████▋                                                                                                                                                                                         | 15/500 [00:01<00:39, 12.31it/s]/home/caio.rhoden/miniconda3/envs/nq/lib/python3.11/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
Training models:  22%|█████████████████████████████████████████▊                                                                                                                                                    | 110/500 [00:08<00:30, 12.91it/s]/home/caio.rhoden/miniconda3/envs/nq/lib/python3.11/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
Training models:  36%|████████████████████████████████████████████████████████████████████▍               

2320

In [22]:
### saving weights as tensores


for exp in range(len(EXPERIMENTS)):
    os.makedirs(f"weights/judge_3k/{EXPERIMENTS[exp]}", exist_ok=True)
    os.makedirs(f"weights/groundtruth_3k/{EXPERIMENTS[exp]}", exist_ok=True)
    torch.save(torch.tensor([arr.coef_[0] if arr is not None else (np.arange(100) < 16).astype(float64) for arr in judge_models[exp]]), f"weights/judge_3k/{EXPERIMENTS[exp]}/weights.pt")
    torch.save(torch.tensor([arr.intercept_ if arr is not None else np.array([[1]]) for arr in judge_models[exp]]), f"weights/judge_3k/{EXPERIMENTS[exp]}/lr_3k_bias.pt")
    torch.save(torch.tensor([arr.coef_[0] if arr is not None else (np.arange(100) < 16).astype(float64) for arr in groundtruth_models[exp]]), f"weights/groundtruth_3k/{EXPERIMENTS[exp]}/weights.pt")
    torch.save(torch.tensor([arr.intercept_ if arr is not None else np.array([[1]]) for arr in groundtruth_models[exp]]), f"weights/groundtruth_3k/{EXPERIMENTS[exp]}/lr_3k_bias.pt")


### saving for datamodels generations
for exp in range(len(EXPERIMENTS)):
    jg_path = f"runs/{EXPERIMENTS[exp]}/datamodels/models/judge_3k/"
    gt_path = f"runs/{EXPERIMENTS[exp]}/datamodels/models/groundtruth_3k/"
    os.makedirs(jg_path, exist_ok=True)
    os.makedirs(gt_path, exist_ok=True)

    torch.save(torch.tensor([arr.coef_[0] if arr is not None else (np.arange(100) < 16).astype(float64) for arr in judge_models[exp]]), f"{jg_path}/0_499_weights.pt")
    torch.save(torch.tensor([arr.intercept_ if arr is not None else np.array([[1]]) for arr in judge_models[exp]]), f"{jg_path}/0_499_bias.pt")
    torch.save(torch.tensor([arr.coef_[0] if arr is not None else (np.arange(100) < 16).astype(float64) for arr in groundtruth_models[exp]]), f"{gt_path}/0_499_weights.pt")
    torch.save(torch.tensor([arr.intercept_ if arr is not None else np.array([[1]]) for arr in groundtruth_models[exp]]), f"{gt_path}/0_499_bias.pt")

/tmp/ipykernel_1899081/665578937.py:8: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  torch.save(torch.tensor([arr.intercept_ if arr is not None else np.array([[1]]) for arr in judge_models[exp]]), f"weights/judge_3k/{EXPERIMENTS[exp]}/lr_3k_bias.pt")
/tmp/ipykernel_1899081/665578937.py:10: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  torch.save(torch.tensor([arr.intercept_ if arr is not None else np.array([[1]]) for arr in groundtruth_models[exp]]), f"weights/groundtruth_3k/{EXPERIMENTS[exp]}/lr_3k_bias.pt")
/tmp/ipykernel_1899081/665578937.py:21: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and wil

In [22]:
# ## EVALUATE 5K MODELS
# LOAD = True
# if LOAD:
#     judge_models = load_logistic_models_for_subfolder("judge_5k", "weights.pt", "bias.pt", experiments=EXPERIMENTS)

X_test_judge, y_test_judge = get_test_sets(df, subfolder="voting_alt1")
X_test_groundtruth, y_test_groundtruth = get_test_sets(df, subfolder="groundtruth")
_eval_df = {
    "experiment": [],
    "type": [],
    "auc": [],
}

for i in range(1):
    for j in range(500):
        # model = judge_models[i][j]
        # if model is not None:
        #     _x = X_test_judge[i][j]
        #     _y = y_test_judge[i][j].ravel()
        #     score = roc_auc_score(_y, model.predict_proba(_x)[:, 1])

        #     _eval_df["experiment"].append(EXPERIMENTS[i])
        #     _eval_df["type"].append("judge")
        #     _eval_df["auc"].append(score)

        model = groundtruth_models[i][j]
        if model is not None:
            _x = X_test_groundtruth[i][j]
            _y = y_test_groundtruth[i][j].ravel()
            score = roc_auc_score(_y, model.predict_proba(_x)[:, 1])

            _eval_df["experiment"].append(EXPERIMENTS[i])
            _eval_df["type"].append("groundtruth")
            _eval_df["auc"].append(score)


eval_df_3k = pl.DataFrame(_eval_df)
print(eval_df_3k.filter(pl.col("auc").is_not_null() & pl.col("auc").is_not_nan()).group_by("type").agg(pl.col("auc").mean()))

/home/caio.rhoden/miniconda3/envs/nq/lib/python3.11/site-packages/sklearn/metrics/_ranking.py:442: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/home/caio.rhoden/miniconda3/envs/nq/lib/python3.11/site-packages/sklearn/metrics/_ranking.py:442: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/home/caio.rhoden/miniconda3/envs/nq/lib/python3.11/site-packages/sklearn/metrics/_ranking.py:442: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/home/caio.rhoden/miniconda3/envs/nq/lib/python3.11/site-packages/sklearn/metrics/_ranking.py:442: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/home/caio.rhoden/miniconda3/envs/nq/lib/python3.11/site-packages/sklearn/metrics/_ranking.py:442: UndefinedMetricWarnin

shape: (1, 2)
┌─────────────┬──────────┐
│ type        ┆ auc      │
│ ---         ┆ ---      │
│ str         ┆ f64      │
╞═════════════╪══════════╡
│ groundtruth ┆ 0.498757 │
└─────────────┴──────────┘


/home/caio.rhoden/miniconda3/envs/nq/lib/python3.11/site-packages/sklearn/metrics/_ranking.py:442: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/home/caio.rhoden/miniconda3/envs/nq/lib/python3.11/site-packages/sklearn/metrics/_ranking.py:442: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/home/caio.rhoden/miniconda3/envs/nq/lib/python3.11/site-packages/sklearn/metrics/_ranking.py:442: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(


In [20]:
_x

array([], dtype=float64)

In [44]:
eval_df_3k.filter(pl.col("type") == "groundtruth")

experiment,type,auc
str,str,f64
"""experiment_1""","""groundtruth""",0.478957
"""experiment_1""","""groundtruth""",0.502207
"""experiment_1""","""groundtruth""",0.516908
"""experiment_1""","""groundtruth""",0.473961
"""experiment_1""","""groundtruth""",0.540178
…,…,…
"""experiment_1""","""groundtruth""",0.543971
"""experiment_1""","""groundtruth""",0.514682
"""experiment_1""","""groundtruth""",0.53355


## 5K Collections Training

In [34]:
X_train_judge_5k, y_train_judge_5k = get_train_sets(df, subfolder="voting_alt1", num_collections=5000)
X_train_groundtruth_5k, y_train_groundtruth_5k = get_train_sets(df, subfolder="groundtruth", num_collections=5000)

judge_models_5k = []
groundtruth_models_5k = []

with tqdm(total=1 * 500, desc="Training models (5k)") as pbar:
    for i in range(1):
        _j_arr, _g_arr = [], []
        
        for j in range(500):
            # Judge model
            X = X_train_judge_5k[i][j]
            y = y_train_judge_5k[i][j].ravel()

            if best_params["judge"][i][j] is None:
                _j_arr.append(None)
            else:
                model = LogisticRegression(**best_params["judge"][i][j], max_iter=2000, random_state=42)
                model.fit(X, y)
                _j_arr.append(model)

            ## Groundtruth model
            X = X_train_groundtruth_5k[i][j]
            y = y_train_groundtruth_5k[i][j].ravel()

            if best_params["groundtruth"][i][j] is None or y.sum() == 0:
                _g_arr.append(None)
            else:
                model = LogisticRegression(**best_params["groundtruth"][i][j], max_iter=2000, random_state=42)
                model.fit(X, y)
                _g_arr.append(model)
            
            pbar.update(1)
        
        judge_models_5k.append(_j_arr)
        groundtruth_models_5k.append(_g_arr)


del X_train_judge_5k, y_train_judge_5k, X_train_groundtruth_5k, y_train_groundtruth_5k
gc.collect()

DEBUG: Split DF shape: (2500000, 7)
DEBUG: Split DF shape: (2500000, 7)


Training models (5k): 100%|██████████████████████████████████████████████████████████████████| 500/500 [01:04<00:00,  7.73it/s]


0

In [35]:
### saving weights as tensores (5k)

for exp in range(len(EXPERIMENTS)):
    os.makedirs(f"weights/judge_5k/{EXPERIMENTS[exp]}", exist_ok=True)
    os.makedirs(f"weights/groundtruth_5k/{EXPERIMENTS[exp]}", exist_ok=True)
    torch.save(torch.tensor([arr.coef_[0] if arr is not None else (np.arange(100) < 16).astype(float64) for arr in judge_models_5k[exp]]), f"weights/judge_5k/{EXPERIMENTS[exp]}/weights.pt")
    torch.save(torch.tensor([arr.intercept_ if arr is not None else np.array([[1]]) for arr in judge_models_5k[exp]]), f"weights/judge_5k/{EXPERIMENTS[exp]}/lr_5k_bias.pt")
    torch.save(torch.tensor([arr.coef_[0] if arr is not None else (np.arange(100) < 16).astype(float64) for arr in groundtruth_models_5k[exp]]), f"weights/groundtruth_5k/{EXPERIMENTS[exp]}/weights.pt")
    torch.save(torch.tensor([arr.intercept_ if arr is not None else np.array([[1]]) for arr in groundtruth_models_5k[exp]]), f"weights/groundtruth_5k/{EXPERIMENTS[exp]}/lr_5k_bias.pt")


### saving for datamodels generations (5k)
for exp in range(len(EXPERIMENTS)):
    jg_path = f"runs/{EXPERIMENTS[exp]}/datamodels/models/judge_5k/"
    gt_path = f"runs/{EXPERIMENTS[exp]}/datamodels/models/groundtruth_5k/"
    os.makedirs(jg_path, exist_ok=True)
    os.makedirs(gt_path, exist_ok=True)

    torch.save(torch.tensor([arr.coef_[0] if arr is not None else (np.arange(100) < 16).astype(float64) for arr in judge_models_5k[exp]]), f"{jg_path}/0_499_weights.pt")
    torch.save(torch.tensor([arr.intercept_ if arr is not None else np.array([[1]]) for arr in judge_models_5k[exp]]), f"{jg_path}/0_499_bias.pt")
    torch.save(torch.tensor([arr.coef_[0] if arr is not None else (np.arange(100) < 16).astype(float64) for arr in groundtruth_models_5k[exp]]), f"{gt_path}/0_499_weights.pt")
    torch.save(torch.tensor([arr.intercept_ if arr is not None else np.array([[1]]) for arr in groundtruth_models_5k[exp]]), f"{gt_path}/0_499_bias.pt")

/tmp/ipykernel_1899081/1851484617.py:7: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  torch.save(torch.tensor([arr.intercept_ if arr is not None else np.array([[1]]) for arr in judge_models_5k[exp]]), f"weights/judge_5k/{EXPERIMENTS[exp]}/lr_5k_bias.pt")
/tmp/ipykernel_1899081/1851484617.py:9: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  torch.save(torch.tensor([arr.intercept_ if arr is not None else np.array([[1]]) for arr in groundtruth_models_5k[exp]]), f"weights/groundtruth_5k/{EXPERIMENTS[exp]}/lr_5k_bias.pt")
/tmp/ipykernel_1899081/1851484617.py:20: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated,

In [36]:
## EVALUATE 5K MODELS
LOAD = True
if LOAD:
    judge_models = load_logistic_models_for_subfolder("judge_5k", "weights.pt", "bias.pt", experiments=EXPERIMENTS)

X_test_judge, y_test_judge = get_test_sets(df, subfolder="voting_alt1")
X_test_groundtruth, y_test_groundtruth = get_test_sets(df, subfolder="groundtruth")

_eval_df = {
    "experiment": [],
    "type": [],
    "auc": [],
}

for i in range(1):
    for j in range(500):
        model = judge_models[i][j]
        if model is not None:
            _x = X_test_judge[i][j]
            _y = y_test_judge[i][j].ravel()
            score = roc_auc_score(_y, model.predict_proba(_x)[:, 1])

            _eval_df["experiment"].append(EXPERIMENTS[i])
            _eval_df["type"].append("judge")
            _eval_df["auc"].append(score)

        model = groundtruth_models_5k[i][j]
        if model is not None:
            _x = X_test_groundtruth[i][j]
            _y = y_test_groundtruth[i][j].ravel()
            score = roc_auc_score(_y, model.predict_proba(_x)[:, 1])

            _eval_df["experiment"].append(EXPERIMENTS[i])
            _eval_df["type"].append("groundtruth")
            _eval_df["auc"].append(score)


eval_df_5k = pl.DataFrame(_eval_df)
print(eval_df_5k.filter(pl.col("auc").is_not_null() & pl.col("auc").is_not_nan()).group_by("type").agg(pl.col("auc").mean()))

DEBUG: Split DF shape: (100000, 7)
DEBUG: Split DF shape: (500000, 7)


/home/caio.rhoden/miniconda3/envs/nq/lib/python3.11/site-packages/sklearn/metrics/_ranking.py:442: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/home/caio.rhoden/miniconda3/envs/nq/lib/python3.11/site-packages/sklearn/metrics/_ranking.py:442: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/home/caio.rhoden/miniconda3/envs/nq/lib/python3.11/site-packages/sklearn/metrics/_ranking.py:442: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/home/caio.rhoden/miniconda3/envs/nq/lib/python3.11/site-packages/sklearn/metrics/_ranking.py:442: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/home/caio.rhoden/miniconda3/envs/nq/lib/python3.11/site-packages/sklearn/metrics/_ranking.py:442: UndefinedMetricWarnin

shape: (2, 2)
┌─────────────┬──────────┐
│ type        ┆ auc      │
│ ---         ┆ ---      │
│ str         ┆ f64      │
╞═════════════╪══════════╡
│ judge       ┆ 0.608404 │
│ groundtruth ┆ 0.501783 │
└─────────────┴──────────┘


/home/caio.rhoden/miniconda3/envs/nq/lib/python3.11/site-packages/sklearn/metrics/_ranking.py:442: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(


## 10K Collections Training

In [37]:
X_train_judge_10k, y_train_judge_10k = get_train_sets(df, subfolder="voting_alt1", num_collections=10000)
X_train_groundtruth_10k, y_train_groundtruth_10k = get_train_sets(df, subfolder="groundtruth", num_collections=10000)

judge_models_10k = []
groundtruth_models_10k = []

with tqdm(total=1 * 500, desc="Training models (10k)") as pbar:
    for i in range(1):
        _j_arr, _g_arr = [], []
        
        for j in range(500):
            # Judge model
            X = X_train_judge_10k[i][j]
            y = y_train_judge_10k[i][j].ravel()

            if best_params["judge"][i][j] is None:
                _j_arr.append(None)
            else:
                model = LogisticRegression(**best_params["judge"][i][j], max_iter=2000, random_state=42)
                model.fit(X, y)
                _j_arr.append(model)

            ## Groundtruth model
            X = X_train_groundtruth_10k[i][j]
            y = y_train_groundtruth_10k[i][j].ravel()

            if best_params["groundtruth"][i][j] is None or y.sum() == 0:
                _g_arr.append(None)
            else:
                model = LogisticRegression(**best_params["groundtruth"][i][j], max_iter=2000, random_state=42)
                model.fit(X, y)
                _g_arr.append(model)
            
            pbar.update(1)
        
        judge_models_10k.append(_j_arr)
        groundtruth_models_10k.append(_g_arr)


del X_train_judge_10k, y_train_judge_10k, X_train_groundtruth_10k, y_train_groundtruth_10k
gc.collect()

DEBUG: Split DF shape: (5000000, 7)
DEBUG: Split DF shape: (5000000, 7)


Training models (10k): 100%|█████████████████████████████████████████████████████████████████| 500/500 [02:00<00:00,  4.14it/s]


0

In [38]:
### saving weights as tensores (10k)

for exp in range(len(EXPERIMENTS)):
    os.makedirs(f"weights/judge_10k/{EXPERIMENTS[exp]}", exist_ok=True)
    os.makedirs(f"weights/groundtruth_10k/{EXPERIMENTS[exp]}", exist_ok=True)
    torch.save(torch.tensor([arr.coef_[0] if arr is not None else (np.arange(100) < 16).astype(float64) for arr in judge_models_10k[exp]]), f"weights/judge_10k/{EXPERIMENTS[exp]}/weights.pt")
    torch.save(torch.tensor([arr.intercept_ if arr is not None else np.array([[1]]) for arr in judge_models_10k[exp]]), f"weights/judge_10k/{EXPERIMENTS[exp]}/lr_10k_bias.pt")
    torch.save(torch.tensor([arr.coef_[0] if arr is not None else (np.arange(100) < 16).astype(float64) for arr in groundtruth_models_10k[exp]]), f"weights/groundtruth_10k/{EXPERIMENTS[exp]}/weights.pt")
    torch.save(torch.tensor([arr.intercept_ if arr is not None else np.array([[1]]) for arr in groundtruth_models_10k[exp]]), f"weights/groundtruth_10k/{EXPERIMENTS[exp]}/lr_10k_bias.pt")


### saving for datamodels generations (10k)
for exp in range(len(EXPERIMENTS)):
    jg_path = f"runs/{EXPERIMENTS[exp]}/datamodels/models/judge_10k/"
    gt_path = f"runs/{EXPERIMENTS[exp]}/datamodels/models/groundtruth_10k/"
    os.makedirs(jg_path, exist_ok=True)
    os.makedirs(gt_path, exist_ok=True)

    torch.save(torch.tensor([arr.coef_[0] if arr is not None else (np.arange(100) < 16).astype(float64) for arr in judge_models_10k[exp]]), f"{jg_path}/0_499_weights.pt")
    torch.save(torch.tensor([arr.intercept_ if arr is not None else np.array([[1]]) for arr in judge_models_10k[exp]]), f"{jg_path}/0_499_bias.pt")
    torch.save(torch.tensor([arr.coef_[0] if arr is not None else (np.arange(100) < 16).astype(float64) for arr in groundtruth_models_10k[exp]]), f"{gt_path}/0_499_weights.pt")
    torch.save(torch.tensor([arr.intercept_ if arr is not None else np.array([[1]]) for arr in groundtruth_models_10k[exp]]), f"{gt_path}/0_499_bias.pt")

/tmp/ipykernel_1899081/3674576619.py:7: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  torch.save(torch.tensor([arr.intercept_ if arr is not None else np.array([[1]]) for arr in judge_models_10k[exp]]), f"weights/judge_10k/{EXPERIMENTS[exp]}/lr_10k_bias.pt")
/tmp/ipykernel_1899081/3674576619.py:9: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  torch.save(torch.tensor([arr.intercept_ if arr is not None else np.array([[1]]) for arr in groundtruth_models_10k[exp]]), f"weights/groundtruth_10k/{EXPERIMENTS[exp]}/lr_10k_bias.pt")
/tmp/ipykernel_1899081/3674576619.py:20: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is depre

In [39]:
## EVALUATE 10K MODELS
LOAD = True
if LOAD:
    judge_models = load_logistic_models_for_subfolder("judge_10k", "weights.pt", "bias.pt", experiments=EXPERIMENTS)

X_test_judge, y_test_judge = get_test_sets(df, subfolder="voting_alt1")
X_test_groundtruth, y_test_groundtruth = get_test_sets(df, subfolder="groundtruth")

_eval_df = {
    "experiment": [],
    "type": [],
    "auc": [],
}

for i in range(1):
    for j in range(500):
        model = judge_models[i][j]
        if model is not None:
            _x = X_test_judge[i][j]
            _y = y_test_judge[i][j].ravel()
            score = roc_auc_score(_y, model.predict_proba(_x)[:, 1])

            _eval_df["experiment"].append(EXPERIMENTS[i])
            _eval_df["type"].append("judge")
            _eval_df["auc"].append(score)

        model = groundtruth_models_10k[i][j]
        if model is not None:
            _x = X_test_groundtruth[i][j]
            _y = y_test_groundtruth[i][j].ravel()
            score = roc_auc_score(_y, model.predict_proba(_x)[:, 1])

            _eval_df["experiment"].append(EXPERIMENTS[i])
            _eval_df["type"].append("groundtruth")
            _eval_df["auc"].append(score)


eval_df_10k = pl.DataFrame(_eval_df)
print(eval_df_10k.filter(pl.col("auc").is_not_null() & pl.col("auc").is_not_nan()).group_by("type").agg(pl.col("auc").mean()))

DEBUG: Split DF shape: (100000, 7)
DEBUG: Split DF shape: (500000, 7)


/home/caio.rhoden/miniconda3/envs/nq/lib/python3.11/site-packages/sklearn/metrics/_ranking.py:442: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/home/caio.rhoden/miniconda3/envs/nq/lib/python3.11/site-packages/sklearn/metrics/_ranking.py:442: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/home/caio.rhoden/miniconda3/envs/nq/lib/python3.11/site-packages/sklearn/metrics/_ranking.py:442: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/home/caio.rhoden/miniconda3/envs/nq/lib/python3.11/site-packages/sklearn/metrics/_ranking.py:442: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/home/caio.rhoden/miniconda3/envs/nq/lib/python3.11/site-packages/sklearn/metrics/_ranking.py:442: UndefinedMetricWarnin

shape: (2, 2)
┌─────────────┬──────────┐
│ type        ┆ auc      │
│ ---         ┆ ---      │
│ str         ┆ f64      │
╞═════════════╪══════════╡
│ groundtruth ┆ 0.501379 │
│ judge       ┆ 0.612661 │
└─────────────┴──────────┘


/home/caio.rhoden/miniconda3/envs/nq/lib/python3.11/site-packages/sklearn/metrics/_ranking.py:442: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(


## 15K Collections Training

In [40]:
X_train_judge_15k, y_train_judge_15k = get_train_sets(df, subfolder="voting_alt1", num_collections=15000)
X_train_groundtruth_15k, y_train_groundtruth_15k = get_train_sets(df, subfolder="groundtruth", num_collections=15000)

judge_models_15k = []
groundtruth_models_15k = []

with tqdm(total=1 * 500, desc="Training models (15k)") as pbar:
    for i in range(1):
        _j_arr, _g_arr = [], []
        
        for j in range(500):
            # Judge model
            X = X_train_judge_15k[i][j]
            y = y_train_judge_15k[i][j].ravel()

            if best_params["judge"][i][j] is None:
                _j_arr.append(None)
            else:
                model = LogisticRegression(**best_params["judge"][i][j], max_iter=2000, random_state=42)
                model.fit(X, y)
                _j_arr.append(model)

            ## Groundtruth model
            X = X_train_groundtruth_15k[i][j]
            y = y_train_groundtruth_15k[i][j].ravel()

            if best_params["groundtruth"][i][j] is None or y.sum() == 0:
                _g_arr.append(None)
            else:
                model = LogisticRegression(**best_params["groundtruth"][i][j], max_iter=2000, random_state=42)
                model.fit(X, y)
                _g_arr.append(model)
            
            pbar.update(1)
        
        judge_models_15k.append(_j_arr)
        groundtruth_models_15k.append(_g_arr)


del X_train_judge_15k, y_train_judge_15k, X_train_groundtruth_15k, y_train_groundtruth_15k
gc.collect()

DEBUG: Split DF shape: (7500000, 7)
DEBUG: Split DF shape: (7500000, 7)


Training models (15k): 100%|█████████████████████████████████████████████████████████████████| 500/500 [03:03<00:00,  2.73it/s]


0

In [41]:
### saving weights as tensores (15k)

for exp in range(len(EXPERIMENTS)):
    os.makedirs(f"weights/judge_15k/{EXPERIMENTS[exp]}", exist_ok=True)
    os.makedirs(f"weights/groundtruth_15k/{EXPERIMENTS[exp]}", exist_ok=True)
    torch.save(torch.tensor([arr.coef_[0] if arr is not None else (np.arange(100) < 16).astype(float64) for arr in judge_models_15k[exp]]), f"weights/judge_15k/{EXPERIMENTS[exp]}/weights.pt")
    torch.save(torch.tensor([arr.intercept_ if arr is not None else np.array([[1]]) for arr in judge_models_15k[exp]]), f"weights/judge_15k/{EXPERIMENTS[exp]}/lr_15k_bias.pt")
    torch.save(torch.tensor([arr.coef_[0] if arr is not None else (np.arange(100) < 16).astype(float64) for arr in groundtruth_models_15k[exp]]), f"weights/groundtruth_15k/{EXPERIMENTS[exp]}/weights.pt")
    torch.save(torch.tensor([arr.intercept_ if arr is not None else np.array([[1]]) for arr in groundtruth_models_15k[exp]]), f"weights/groundtruth_15k/{EXPERIMENTS[exp]}/lr_15k_bias.pt")


### saving for datamodels generations (15k)
for exp in range(len(EXPERIMENTS)):
    jg_path = f"runs/{EXPERIMENTS[exp]}/datamodels/models/judge_15k/"
    gt_path = f"runs/{EXPERIMENTS[exp]}/datamodels/models/groundtruth_15k/"
    os.makedirs(jg_path, exist_ok=True)
    os.makedirs(gt_path, exist_ok=True)

    torch.save(torch.tensor([arr.coef_[0] if arr is not None else (np.arange(100) < 16).astype(float64) for arr in judge_models_15k[exp]]), f"{jg_path}/0_499_weights.pt")
    torch.save(torch.tensor([arr.intercept_ if arr is not None else np.array([[1]]) for arr in judge_models_15k[exp]]), f"{jg_path}/0_499_bias.pt")
    torch.save(torch.tensor([arr.coef_[0] if arr is not None else (np.arange(100) < 16).astype(float64) for arr in groundtruth_models_15k[exp]]), f"{gt_path}/0_499_weights.pt")
    torch.save(torch.tensor([arr.intercept_ if arr is not None else np.array([[1]]) for arr in groundtruth_models_15k[exp]]), f"{gt_path}/0_499_bias.pt")

/tmp/ipykernel_1899081/3974615751.py:7: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  torch.save(torch.tensor([arr.intercept_ if arr is not None else np.array([[1]]) for arr in judge_models_15k[exp]]), f"weights/judge_15k/{EXPERIMENTS[exp]}/lr_15k_bias.pt")
/tmp/ipykernel_1899081/3974615751.py:9: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  torch.save(torch.tensor([arr.intercept_ if arr is not None else np.array([[1]]) for arr in groundtruth_models_15k[exp]]), f"weights/groundtruth_15k/{EXPERIMENTS[exp]}/lr_15k_bias.pt")
/tmp/ipykernel_1899081/3974615751.py:20: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is depre

In [42]:
## EVALUATE 15K MODELS
LOAD = True
if LOAD:
    judge_models = load_logistic_models_for_subfolder("judge_15k", "weights.pt", "bias.pt", experiments=EXPERIMENTS)

X_test_judge, y_test_judge = get_test_sets(df, subfolder="voting_alt1")
X_test_groundtruth, y_test_groundtruth = get_test_sets(df, subfolder="groundtruth")

_eval_df = {
    "experiment": [],
    "type": [],
    "auc": [],
}

for i in range(1):
    for j in range(500):
        model = judge_models[i][j]
        if model is not None:
            _x = X_test_judge[i][j]
            _y = y_test_judge[i][j].ravel()
            score = roc_auc_score(_y, model.predict_proba(_x)[:, 1])

            _eval_df["experiment"].append(EXPERIMENTS[i])
            _eval_df["type"].append("judge")
            _eval_df["auc"].append(score)

        model = groundtruth_models_15k[i][j]
        if model is not None:
            _x = X_test_groundtruth[i][j]
            _y = y_test_groundtruth[i][j].ravel()
            score = roc_auc_score(_y, model.predict_proba(_x)[:, 1])

            _eval_df["experiment"].append(EXPERIMENTS[i])
            _eval_df["type"].append("groundtruth")
            _eval_df["auc"].append(score)


eval_df_15k = pl.DataFrame(_eval_df)
print(eval_df_15k.filter(pl.col("auc").is_not_null() & pl.col("auc").is_not_nan()).group_by("type").agg(pl.col("auc").mean()))

DEBUG: Split DF shape: (100000, 7)
DEBUG: Split DF shape: (500000, 7)


/home/caio.rhoden/miniconda3/envs/nq/lib/python3.11/site-packages/sklearn/metrics/_ranking.py:442: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/home/caio.rhoden/miniconda3/envs/nq/lib/python3.11/site-packages/sklearn/metrics/_ranking.py:442: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/home/caio.rhoden/miniconda3/envs/nq/lib/python3.11/site-packages/sklearn/metrics/_ranking.py:442: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/home/caio.rhoden/miniconda3/envs/nq/lib/python3.11/site-packages/sklearn/metrics/_ranking.py:442: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/home/caio.rhoden/miniconda3/envs/nq/lib/python3.11/site-packages/sklearn/metrics/_ranking.py:442: UndefinedMetricWarnin

shape: (2, 2)
┌─────────────┬──────────┐
│ type        ┆ auc      │
│ ---         ┆ ---      │
│ str         ┆ f64      │
╞═════════════╪══════════╡
│ judge       ┆ 0.615924 │
│ groundtruth ┆ 0.500686 │
└─────────────┴──────────┘


/home/caio.rhoden/miniconda3/envs/nq/lib/python3.11/site-packages/sklearn/metrics/_ranking.py:442: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(


## 20K Collections Training

In [43]:
X_train_judge_20k, y_train_judge_20k = get_train_sets(df, subfolder="voting_alt1", num_collections=20000)
X_train_groundtruth_20k, y_train_groundtruth_20k = get_train_sets(df, subfolder="groundtruth", num_collections=20000)

judge_models_20k = []
groundtruth_models_20k = []

with tqdm(total=1 * 500, desc="Training models (20k)") as pbar:
    for i in range(1):
        _j_arr, _g_arr = [], []
        
        for j in range(500):
            # Judge model
            X = X_train_judge_20k[i][j]
            y = y_train_judge_20k[i][j].ravel()

            if best_params["judge"][i][j] is None:
                _j_arr.append(None)
            else:
                model = LogisticRegression(**best_params["judge"][i][j], max_iter=2000, random_state=42)
                model.fit(X, y)
                _j_arr.append(model)

            ## Groundtruth model
            X = X_train_groundtruth_20k[i][j]
            y = y_train_groundtruth_20k[i][j].ravel()

            if best_params["groundtruth"][i][j] is None or y.sum() == 0:
                _g_arr.append(None)
            else:
                model = LogisticRegression(**best_params["groundtruth"][i][j], max_iter=2000, random_state=42)
                model.fit(X, y)
                _g_arr.append(model)
            
            pbar.update(1)
        
        judge_models_20k.append(_j_arr)
        groundtruth_models_20k.append(_g_arr)


del X_train_judge_20k, y_train_judge_20k, X_train_groundtruth_20k, y_train_groundtruth_20k
gc.collect()

DEBUG: Split DF shape: (10000000, 7)
DEBUG: Split DF shape: (10000000, 7)


Training models (20k): 100%|█████████████████████████████████████████████████████████████████| 500/500 [04:15<00:00,  1.96it/s]


0

In [44]:
### saving weights as tensores (20k)

for exp in range(len(EXPERIMENTS)):
    os.makedirs(f"weights/judge_20k/{EXPERIMENTS[exp]}", exist_ok=True)
    os.makedirs(f"weights/groundtruth_20k/{EXPERIMENTS[exp]}", exist_ok=True)
    torch.save(torch.tensor([arr.coef_[0] if arr is not None else (np.arange(100) < 16).astype(float64) for arr in judge_models_20k[exp]]), f"weights/judge_20k/{EXPERIMENTS[exp]}/weights.pt")
    torch.save(torch.tensor([arr.intercept_ if arr is not None else np.array([[1]]) for arr in judge_models_20k[exp]]), f"weights/judge_20k/{EXPERIMENTS[exp]}/lr_20k_bias.pt")
    torch.save(torch.tensor([arr.coef_[0] if arr is not None else (np.arange(100) < 16).astype(float64) for arr in groundtruth_models_20k[exp]]), f"weights/groundtruth_20k/{EXPERIMENTS[exp]}/weights.pt")
    torch.save(torch.tensor([arr.intercept_ if arr is not None else np.array([[1]]) for arr in groundtruth_models_20k[exp]]), f"weights/groundtruth_20k/{EXPERIMENTS[exp]}/lr_20k_bias.pt")


### saving for datamodels generations (20k)
for exp in range(len(EXPERIMENTS)):
    jg_path = f"runs/{EXPERIMENTS[exp]}/datamodels/models/judge_20k/"
    gt_path = f"runs/{EXPERIMENTS[exp]}/datamodels/models/groundtruth_20k/"
    os.makedirs(jg_path, exist_ok=True)
    os.makedirs(gt_path, exist_ok=True)

    torch.save(torch.tensor([arr.coef_[0] if arr is not None else (np.arange(100) < 16).astype(float64) for arr in judge_models_20k[exp]]), f"{jg_path}/0_499_weights.pt")
    torch.save(torch.tensor([arr.intercept_ if arr is not None else np.array([[1]]) for arr in judge_models_20k[exp]]), f"{jg_path}/0_499_bias.pt")
    torch.save(torch.tensor([arr.coef_[0] if arr is not None else (np.arange(100) < 16).astype(float64) for arr in groundtruth_models_20k[exp]]), f"{gt_path}/0_499_weights.pt")
    torch.save(torch.tensor([arr.intercept_ if arr is not None else np.array([[1]]) for arr in groundtruth_models_20k[exp]]), f"{gt_path}/0_499_bias.pt")

/tmp/ipykernel_1899081/3754046166.py:7: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  torch.save(torch.tensor([arr.intercept_ if arr is not None else np.array([[1]]) for arr in judge_models_20k[exp]]), f"weights/judge_20k/{EXPERIMENTS[exp]}/lr_20k_bias.pt")
/tmp/ipykernel_1899081/3754046166.py:9: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  torch.save(torch.tensor([arr.intercept_ if arr is not None else np.array([[1]]) for arr in groundtruth_models_20k[exp]]), f"weights/groundtruth_20k/{EXPERIMENTS[exp]}/lr_20k_bias.pt")
/tmp/ipykernel_1899081/3754046166.py:20: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is depre

In [45]:
## EVALUATE 20K MODELS
LOAD = True
if LOAD:
    judge_models = load_logistic_models_for_subfolder("judge_20k", "weights.pt", "bias.pt", experiments=EXPERIMENTS)

X_test_judge, y_test_judge = get_test_sets(df, subfolder="voting_alt1")
X_test_groundtruth, y_test_groundtruth = get_test_sets(df, subfolder="groundtruth")

_eval_df = {
    "experiment": [],
    "type": [],
    "auc": [],
}

for i in range(1):
    for j in range(500):
        model = judge_models[i][j]
        if model is not None:
            _x = X_test_judge[i][j]
            _y = y_test_judge[i][j].ravel()
            score = roc_auc_score(_y, model.predict_proba(_x)[:, 1])

            _eval_df["experiment"].append(EXPERIMENTS[i])
            _eval_df["type"].append("judge")
            _eval_df["auc"].append(score)

        model = groundtruth_models_20k[i][j]
        if model is not None:
            _x = X_test_groundtruth[i][j]
            _y = y_test_groundtruth[i][j].ravel()
            score = roc_auc_score(_y, model.predict_proba(_x)[:, 1])

            _eval_df["experiment"].append(EXPERIMENTS[i])
            _eval_df["type"].append("groundtruth")
            _eval_df["auc"].append(score)


eval_df_20k = pl.DataFrame(_eval_df)
print(eval_df_20k.filter(pl.col("auc").is_not_null() & pl.col("auc").is_not_nan()).group_by("type").agg(pl.col("auc").mean()))

DEBUG: Split DF shape: (100000, 7)
DEBUG: Split DF shape: (500000, 7)


/home/caio.rhoden/miniconda3/envs/nq/lib/python3.11/site-packages/sklearn/metrics/_ranking.py:442: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/home/caio.rhoden/miniconda3/envs/nq/lib/python3.11/site-packages/sklearn/metrics/_ranking.py:442: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/home/caio.rhoden/miniconda3/envs/nq/lib/python3.11/site-packages/sklearn/metrics/_ranking.py:442: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/home/caio.rhoden/miniconda3/envs/nq/lib/python3.11/site-packages/sklearn/metrics/_ranking.py:442: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/home/caio.rhoden/miniconda3/envs/nq/lib/python3.11/site-packages/sklearn/metrics/_ranking.py:442: UndefinedMetricWarnin

shape: (2, 2)
┌─────────────┬──────────┐
│ type        ┆ auc      │
│ ---         ┆ ---      │
│ str         ┆ f64      │
╞═════════════╪══════════╡
│ groundtruth ┆ 0.500137 │
│ judge       ┆ 0.617661 │
└─────────────┴──────────┘


/home/caio.rhoden/miniconda3/envs/nq/lib/python3.11/site-packages/sklearn/metrics/_ranking.py:442: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
